# Autograd - Build
This is the build section of the project, where new functions, classes, etc are created and tested. Finished peices are moved into main where they are then called and used by build.

### Rules:
- No ai used to generate any code, only for reaserch, explinations and code reviews
- Finished peices must be abstract and state clearly if any cases are not covered

## Setup

In [19]:
import numpy as np
import main
from typing import Union, List
from matplotlib import pyplot as plt
import matplotlib_inline
import pandas as pd
%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


## Build

In [20]:
class Model():
    """
    
    """
    def __init__(self,
                  input_size : int, hidden_size : int, output_size : int,
                  number_of_layers : int, activation_function, normalisation_function,
                  precision: str = 'float32', random_seed: int = None):
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.number_of_layers = number_of_layers

        self.precision = precision
        self.random_seed = random_seed

        self.activation_function = activation_function
        self.normalisation_function = normalisation_function

        self.activation_function_derivative = main.derivatives.get(
            getattr(self.activation_function, "__name__", None)
        )
        self.normalisation_function_derivative = main.derivatives.get(
            getattr(self.normalisation_function, "__name__", None)
        )


        layers = []
        layer_gradients = {}
        if self.number_of_layers == 1:
            layers.append([main.LinearLayer(self.input_size, self.output_size,
                                            self.precision, self.random_seed)])
            layer_gradients["Layer 1"] = {}
        else:
            layers.append([main.LinearLayer(self.input_size, self.hidden_size,
                                            self.precision, self.random_seed)])
            layer_gradients["Layer 1"] = {}

            for i in range(max(0, self.number_of_layers - 2)):
                layers.append([
                    main.LinearLayer(self.hidden_size, self.hidden_size,
                                     self.precision, self.random_seed),
                    self.activation_function
                ])
                layer_gradients[f"Layer {i+1}"] = {}

            layers.append([
                main.LinearLayer(self.hidden_size, self.output_size,
                                 self.precision, self.random_seed)
            ])
            layer_gradients[f"Layer {self.number_of_layers}"] = {}

        layers.append([self.normalisation_function])

        self.modules = layers
        # self.gradients serves no functional purposse, but make it easier to acces all the differnt layer's gradients at once.
        self.gradients = layer_gradients

        parramaters = {}
        i = 1
        for obj in self.modules:
            for sub_obj in obj:
                if isinstance(sub_obj, main.LinearLayer):
                    parramaters[f"Layer {i}"] = sub_obj.parramaters
                    i += 1

        self.parramaters = parramaters

        self.last_operation = None


    def forward(self, x: Union[np.ndarray, List, float, int]) -> np.ndarray:
        for module in self.modules:
            for obj in module:
                if hasattr(obj, "forward") and callable(obj.forward):
                    x = obj.forward(x)
                elif callable(obj):
                    x = obj(x)
                else:
                    raise TypeError(f"Module - {type(obj).__name__} - does not have a 'forward' function, or is not callable and so is not supported")
        self.last_operation = "forward"
        return x


    def backwards(self, output : np.ndarray, target : np.ndarray, loss_function):
        """
        Backpropagates the output error through the network.

        Notes:
            This implementation supports the common combination of Softmax activation
            followed by mean cross-entropy loss. In that case the derivative with
            respect to the logits is (softmax_output - target) / batch_size.
        """
        if self.last_operation != "forward":
            raise main.SequenceError("Must compleate a forwards pass imidatley before a backwards pass.")

        try:
            output = np.asarray(output, dtype=self.precision)
            target = np.asarray(target, dtype=self.precision)
        except (TypeError, ValueError) as e:
            raise TypeError(f"The inputed arrays - {output} and {target} must be a numeric array or array-like object. Original error: {e}")

        if output.size == 0 or target.size == 0:
            raise ValueError("At least one input is empty.")

        if output.ndim == 1:
            output = np.array([output], dtype=self.precision)

        if target.ndim == 1:
            target = np.array([target], dtype=self.precision)

        if output.shape != target.shape:
            raise ValueError(f"Output shape {output.shape} does not match target shape {target.shape}.")

        # For Softmax + mean CrossEntropyLoss, the derivative with respect to logits is
        # (softmax_output - target) / batch_size. This matches the loss scaling used in
        # CrossEntropyLoss.
        if self.normalisation_function is main.Softmax and loss_function is main.CrossEntropyLoss:
            passed_down_grad = (output - target) / output.shape[0]
        else:
            raise TypeError(
                f"This model's normalisation function - {self.normalisation_function} - currently has no programed back propogtion rules."
            )

        for layer_index in range(self.number_of_layers, 0, -1):
            layer = self.modules[layer_index - 1][0]
            layer.backwards(passed_down_grad)

            self.gradients[f"Layer {layer_index}"] = layer.gradients

            if len(self.modules[layer_index - 1]) > 1:
                if self.activation_function is main.ReLU:
                    relu_mask = main.backwards_ReLU(layer.last_outputed_array)
                    passed_down_grad = layer.passed_down_grad * relu_mask
                else:
                    passed_down_grad = layer.passed_down_grad
            else:
                passed_down_grad = layer.passed_down_grad

        self.gradients[f"Layer {layer_index}"] = layer.gradients
        self.last_operation = "backwards"


    def update_parramaters(self, learning_rate : float):
        """
        """
        if self.last_operation != "backwards":
            raise main.SequenceError("Must compleate a backwards pass imidatley before updating a models parramaters.")

        try: 
            float(learning_rate)
        except (TypeError, ValueError) as exc:
            raise TypeError("Learning rate must be a number, preferably float") from exc

        if not np.isfinite(learning_rate):
            raise ValueError(f"Learning rate must be finite, currently it is - {learning_rate}")

        
        for layer_index in range(self.number_of_layers, 0, -1):
            layer_modual = self.modules[layer_index -1]
            layer = layer_modual[0]
            layer.update_parameters(learning_rate)

        self.last_operation = "update"
        

    def set_parramaters(self, parramaters : dict):
        """
        Sets the model parameters to the ones stored in the input dictionary.

        Args:
            parramaters (dict): The dictionary storing the parramaters.

        Updates:
            self.parramaters: Sets self.parramaters["Layer i"] to parramaters["Layer i"]

        Rasies:
            TypeError: If the input is not a dictionary.
            KeyError: If the inputed dictionary does not have the keys "Layer 1" thorugh "Layer i".
        """

        if not isinstance(parramaters, dict):
            raise TypeError(f"Input - {parramaters} - must be a dict.")

        if set(parramaters.keys()) != set(self.parramaters.keys()):
            raise KeyError(f"Input - {parramaters} - keys do not match those in self.parramaters.")

        if len(self.parramaters) != len(parramaters):
            raise ValueError(f"Inputed dictionary has a different number of elements than the existing dictionary.")

        
        modules_copy = self.modules
        layer_index = 1
        for obj in modules_copy:
            if isinstance(obj, main.LinearLayer):
                key = f"Layer {layer_index}"
                if key not in parramaters:
                    raise KeyError(f"Missing parameter set for {key}.")
                obj.set_parramaters(parramaters[key])
                layer_index += 1

        self.modules = modules_copy
        self.parramaters = parramaters


In [21]:
model = Model(input_size=3, output_size=2, hidden_size=2,
              number_of_layers=4, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=1)
print(model.activation_function_derivative(-1))

0


In [22]:
# Organising the data
# Keep samples as rows: (num_samples, 784)
# Labels as 1D: (num_samples,)
# One-hot labels will then be (num_samples, 10)

data = pd.read_csv(r"C:\Users\acart\OneDrive\Desktop\CodingProjects\mnist\mnist_train.csv")

data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:30000]
Y_dev = data_dev[:, 0]
X_dev = data_dev[:, 1:] / 255.0

data_train = data[30000:m]
Y_train = data_train[:, 0]
X_train = data_train[:, 1:] / 255.0

print(X_train.shape)
print(Y_train.shape)
_, m_train = X_train.shape

(29999, 784)
(29999,)


In [23]:
# one hot gives: (num_samples, 10)
def one_hot(Y):
    Y = np.asarray(Y, dtype=int)
    one_hot_Y = np.zeros((Y.size, Y.max() + 1), dtype=np.float32)
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y

Y_train = one_hot(Y_train)

In [24]:
print(X_train.shape)   # should be (num_samples, 784)
print(Y_train.shape)   # should be (num_samples, 10)

(29999, 784)
(29999, 10)


In [34]:
model = Model(input_size=784, output_size=10, hidden_size=16,
              number_of_layers=3, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=0)


In [ ]:
for i in range(500):
    output = model.forward(X_train)
    loss = main.CrossEntropyLoss(output, Y_train)
    model.backwards(output, Y_train, main.CrossEntropyLoss)
    model.update_parramaters(1e-2)
    if i % 10 == 0: 
        print(f"Epoch: {i+1}    Loss: {loss}    Accuracy: {round(np.exp(-loss) * 100, 4)}%")

Epoch: 1    Loss: 5.3595995961269765    Accuracy: 0.47028%
Epoch: 11    Loss: 4.709164324928679    Accuracy: 0.90123%
Epoch: 21    Loss: 4.2279396579977    Accuracy: 1.45824%
Epoch: 31    Loss: 3.864123191451016    Accuracy: 2.09813%
Epoch: 41    Loss: 3.5861031649217154    Accuracy: 2.77061%
Epoch: 51    Loss: 3.3724345740046613    Accuracy: 3.4306%
Epoch: 61    Loss: 3.2046250382895614    Accuracy: 4.05741%
Epoch: 71    Loss: 3.0698903028024427    Accuracy: 4.64262%
Epoch: 81    Loss: 2.9590042417489753    Accuracy: 5.18705%
Epoch: 91    Loss: 2.864962484547269    Accuracy: 5.69853%
Epoch: 101    Loss: 2.7845069153042497    Accuracy: 6.17595%
Epoch: 111    Loss: 2.714303452961449    Accuracy: 6.62511%
Epoch: 121    Loss: 2.652157341934657    Accuracy: 7.0499%
Epoch: 131    Loss: 2.597709067306393    Accuracy: 7.44439%
Epoch: 141    Loss: 2.548859536355362    Accuracy: 7.81708%
Epoch: 151    Loss: 2.504107496365339    Accuracy: 8.17485%
Epoch: 161    Loss: 2.4631158624758305    Accura